## 1. Imports and Global Constants

This section imports all necessary Python libraries.

In [1]:
import torch
import torch.nn as nn
import math
import copy
import random
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, random_split, Dataset, Subset
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

## 2. Utility Functions

This section includes helper functions that are not part of the core model but are essential for the overall pipeline, such as determining the correct data directory based on the execution environment.

In [3]:
# --- UTILITY FUNCTIONS ---

def get_data_dir():
    """
    Determines the correct data directory based on the execution environment.
    This function makes the code portable between Colab, local PyCharm, and cluster.
    """
    # Default to current working directory for local/cluster
    data_directory = os.path.join(os.getcwd(), 'data/')

    # Attempt to detect Google Colab and use Google Drive path
    try:
        from google.colab import drive
        drive.mount('/content/gdrive')
        # This path assumes your 'DnARnAProject' folder is directly in 'My Drive'
        google_drive_project_path = '/content/gdrive/MyDrive/DnARnAProject/'
        data_directory = os.path.join(google_drive_project_path, 'data/')
        print("Detected Google Colab environment. Using Google Drive path.")
    except ImportError:
        print("Not in Google Colab. Using local/cluster path.")

    if not os.path.isdir(data_directory):
        print(f"Error: The data directory '{data_directory}' does not exist.")
        print("Please ensure your data is located correctly (e.g., in a 'data/' folder relative to your script, or in Google Drive).")
        exit() # Exit if data directory is not found
    return data_directory

## 3. Data Exploration: Loading and Peeking at Raw Data Files

This section loads the primary data files (`data.npz`, `regions.parquet`, and `ensemble_annotation.gff3`) and prints a snippet of their contents to understand their structure and content.

In [4]:
import numpy as np
import pandas as pd
import os

# Ensure get_data_dir() is accessible or defined if this cell is run independently
# If running the full notebook, it will be defined in Section 2.
try:
    data_dir = get_data_dir()
except NameError:
    print("get_data_dir() not found. Please ensure Section 2 has been run or define it manually.")
    # Fallback for manual definition if needed for standalone execution of this cell
    def get_data_dir():
        local_or_cluster_project_path = os.getcwd()
        data_directory = os.path.join(local_or_cluster_project_path, 'data/')
        try:
            from google.colab import drive
            drive.mount('/content/gdrive')
            google_drive_project_path = '/content/gdrive/MyDrive/DnARnAProject/'
            data_directory = os.path.join(google_drive_project_path, 'data/')
            print("Detected Google Colab environment. Using Google Drive path.")
        except ImportError:
            print("Not in Google Colab. Using local/cluster path.")
        if not os.path.isdir(data_directory):
            raise FileNotFoundError(f"Error: The data directory '{data_directory}' does not exist. "
                                    f"Please ensure your data is located correctly for your environment.")
        return data_directory
    data_dir = get_data_dir()


print("--- Loading data.npz ---")
try:
    data_npz_path = os.path.join(data_dir, 'data.npz')
    data_npz = np.load(data_npz_path, allow_pickle=True)
    print("Keys in data.npz:", data_npz.files)
    if 'sequence' in data_npz.files:
        print("First 10 elements of 'sequence' array:")
        print(data_npz['sequence'][:10])
    if 'expressed_plus' in data_npz.files:
        print("First 10 elements of 'expressed_plus' array:")
        print(data_npz['expressed_plus'][:10])
    if 'expressed_minus' in data_npz.files:
        print("First 10 elements of 'expressed_minus' array:")
        print(data_npz['expressed_minus'][:10])
    data_npz.close()
except FileNotFoundError:
    print(f"Error: data.npz not found at {data_npz_path}")
except Exception as e:
    print(f"Error loading data.npz: {e}")

print("\n--- Loading regions.parquet ---")
try:
    regions_parquet_path = os.path.join(data_dir, 'regions.parquet')
    regions_df = pd.read_parquet(regions_parquet_path)
    print("Head of regions.parquet:")
    print(regions_df.head())
    print("\nColumns in regions.parquet:", regions_df.columns.tolist())
except FileNotFoundError:
    print(f"Error: regions.parquet not found at {regions_parquet_path}")
except Exception as e:
    print(f"Error loading regions.parquet: {e}")

print("\n--- Loading ensembl_annotation.gff3 (first 10 lines) ---")
try:
    gff_path = os.path.join(data_dir, 'ensembl_annotation.gff3')
    with open(gff_path, 'r') as f:
        for i, line in enumerate(f):
            print(line.strip())
            if i >= 9: # Print first 10 lines
                break
except FileNotFoundError:
    print(f"Error: ensembl_annotation.gff3 not found at {gff_path}. Please ensure the file exists in the data directory.")
except Exception as e:
    print(f"Error loading ensembl_annotation.gff3: {e}")


Not in Google Colab. Using local/cluster path.
--- Loading data.npz ---
Keys in data.npz: ['sequence', 'expressed_plus', 'expressed_minus']
First 10 elements of 'sequence' array:
[1 1 0 1 0 1 1 0 1 0]
First 10 elements of 'expressed_plus' array:
[0 0 0 0 0 0 0 0 0 0]
First 10 elements of 'expressed_minus' array:
[0 0 0 0 0 0 0 0 0 0]

--- Loading regions.parquet ---
Head of regions.parquet:
  contig strand  start  offset  window_size  num_expressed
0   chrI      +   7655    7655         2048            205
1   chrI      +   7656    7656         2048            206
2   chrI      +   7657    7657         2048            207
3   chrI      +   7658    7658         2048            208
4   chrI      +   7659    7659         2048            209

Columns in regions.parquet: ['contig', 'strand', 'start', 'offset', 'window_size', 'num_expressed']

--- Loading ensembl_annotation.gff3 (first 10 lines) ---
##gff-version 3
##sequence-region   I 1 230218
##sequence-region   II 1 813184
##sequence-reg